In [1]:
import os
from pyflink.table import EnvironmentSettings, TableEnvironment, TableDescriptor, Schema, DataTypes

In [2]:
table_environment = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())
configration = table_environment.get_config().get_configuration()
configration.set_string("execution.target", "remote")
configration.set_string("rest.address", "flink-jobmanager")
configration.set_integer("rest.port", 8082)
configration.set_string("execution.checkpointing.interval", "3 s")
configration.set_string("execution.checkpointing.mode", "EXACTLY_ONCE")
configration.set_string("execution.checkpointing.timeout", "10 min")
table_environment.get_config().get_configuration().set_string("parallelism.default", "1")

## Mysql -> Debezium -> Kafka -> Flink Source Table

In [3]:
table_environment.execute_sql("CREATE DATABASE IF NOT EXISTS mmix_mysql_cdc;")
table_environment.execute_sql("USE mmix_mysql_cdc;")

In [4]:
kafka_source_descriptor = (
    TableDescriptor
    .for_connector("kafka")
    .schema(Schema.new_builder()
            .column("id", DataTypes.INT())
            .column("name", DataTypes.STRING())
            .column("department", DataTypes.INT())
            .column("salary", DataTypes.DOUBLE())
            .build())
    .option("topic", "debezium.mysql.source.mmix.employees")
    .option("properties.bootstrap.servers", "confluent-kafka-broker:19092")
    .option("properties.group.id", "mmix-debezium-mysql-source-flink")
    .option("scan.startup.mode", "earliest-offset")
    .option("format", "avro-confluent")
    .option("avro-confluent.schema-registry.url", "http://confluent-schema-registry:8081")
    .build())
table_environment.create_table("mmix_employees_kafka_source", kafka_source_descriptor)
#table_environment.execute_sql("SELECT id, name, department, salary FROM mmix_employees_kafka_source")

In [ ]:
print_sink_descriptor = (
    TableDescriptor
    .for_connector("print")
    .schema(
        Schema.new_builder()
        .column("id", DataTypes.INT())
        .column("name", DataTypes.STRING())
        .column("department", DataTypes.INT())
        .column("salary", DataTypes.DOUBLE())
        .build()
    )
    .build()
)

table_environment.create_table("mmix_employees_print_sink", print_sink_descriptor)

In [5]:
table_environment.execute_sql(
    """
    CREATE TABLE mmix_employees_iceberg_sink
    (
        id         INT,
        name       STRING,
        department INT,
        salary DOUBLE,
        PRIMARY KEY (id) NOT ENFORCED
    ) WITH (
          'connector' = 'iceberg',
          'catalog-name' = 'iceberg_catalog',
          'catalog-type' = 'hadoop',
          'warehouse' = 's3a://mmix-prod-dataengineer-datalakehouse/streaming',
          'format-version' = '2'
          );
    """
)
table_environment.execute_sql("INSERT INTO mmix_employees_iceberg_sink SELECT id, name, department, salary FROM mmix_employees_kafka_source;").print()